In [1]:
import pandas as pd
import re
import jieba
import nltk
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords as nltk_stopwords
from stopwords import get_stopwords

# 1- Setup 



In [2]:
#nltk.data.path.append('/Users/rachelliu/nltk_data')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/rachelliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rachelliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2- Load CSV files 

In [3]:
# Load CSV files
df_en = pd.read_csv('data/Data_English.csv')
df_ar = pd.read_csv('data/Data_Arabic.csv')
df_zh = pd.read_csv('data/Data_Mandarin.csv')
df_fa = pd.read_csv('data/Data_Farsi.csv')
df_fr = pd.read_csv('data/Data_French.csv')
df_hi = pd.read_csv('data/Data_Hindi.csv')
df_id = pd.read_csv('data/Data_Indonesian.csv')
df_pt = pd.read_csv('data/Data_Portugese.csv')
df_ru = pd.read_csv('data/Data_Russian.csv')
df_es = pd.read_csv('data/Data_Spanish.csv')
df_tr = pd.read_csv('data/Data_Turkish.csv')
df_uk = pd.read_csv('data/Data_Ukranian.csv')
df_ur = pd.read_csv('data/Data_Urdu.csv')

# 3- Combine ALl Data

In [4]:
df = pd.concat([df_en, df_ar, df_zh, df_fa, df_fr, df_hi, df_id, df_pt, df_ru, df_es, df_tr, df_uk, df_ur], ignore_index=True)

# 4- NLTK Stopwords Map 

In [5]:
nltk_lang_map = {
    'en': 'english', 'ar': 'arabic', 'es': 'spanish', 'pt': 'portuguese',
    'fr': 'french', 'ru': 'russian', 'tr': 'turkish', 'id': 'indonesian'
}

nltk_stopwords_dict = {
    lang_code: set(nltk_stopwords.words(nltk_lang_map[lang_code]))
    for lang_code in nltk_lang_map
}


# 5- Extended Stopwords 

In [6]:
# Arabic (ar)
extended_arabic_stopwords = set(nltk_stopwords.words('arabic')).union({
    'التي', 'الذي', 'الذين', 'اللذان', 'اللذين', 'اللائي', 'اللاتي',
    'كأن', 'لعل', 'حتى', 'إذن', 'قد', 'لن', 'لم', 'ما', 'من', 'في',
    'عن', 'على', 'إلى', 'أن', 'إن', 'لكن', 'أو', 'بل', 'ثم', 'ف', 'و', 'أم'
})

# Mandarin (zh)
extended_mandarin_stopwords = set("""
的 了 和 是 就 都 而 及 與 著 或 一個 沒有 我們 你們 他們 她們 是否 所以 而且 並且 如果 但是 因為 這樣 這個 那個 以及 啊 嗎 吧 呢 嘛 哦 哇
也 很 被 要 在 有 就是 就算 誰 哪 那 什麼 怎麼 怎樣 因為 此 此外 仍然 每個 對於 通過
""".split())

# Farsi (fa)
extended_farsi_stopwords = set("""
اما اگر از اش با باشد باشدند بودن بود بودم بودند باشد بکن بکنید بکند برای برسد بس باشد بتوان بدون بعضی بعد به بی تا تمام تنها توج توجه تیز ترین ترین ها هاست هایی همچنین حتی خاص خود خیلی خیلیی در دریافت دارد داشت داشتم داشتند داشته در دیگر را راستی راحت راحتی راه روز روزی روی شد شدند شده شدن شدم شدید شدیدا شوید شاید شود شون شناسی صورت طبق طول طی ظهر ظهرها عجب عجیب علی رغم علیه عمل فقط فرا فرق فوق قبل قبلا قد قدری قرار قطعا لحظه لذا لذاست لذاً لک لیک لیل مان مانم مانید مانیم مانند مارس مثلا مثال مثالاً مثل مثلاً مدتی مردم مرسی مشکل مشکلات مطمئنا مطرح معین مشخص معلومات معکوس معلوم مفید مقداری ممکن می میخواهم میتوانم میکند نا ناگه نداشت ندارد نماید نمی نمیخواهید نمیخواهیم نمیخواهم نمیخواهند نمی‌کنید نیست نیستند نیک هنگامی واکنش واکنشی وجود واقعی واقعیاً واقعیت واقعاً ولی وی نیز یک
""".split())

# Urdu (ur)
extended_urdu_stopwords = set("""
اور کے کو میں یہ کہ ہے ہوں تھا تھے ہیں نہیں پر سے بھی نے تو تھا تک کی کریں کرو کر رہا رہی رہے جیسے جب جس جن جنہیں جنہوں جو چکا چکی چکے کیا کون کسی کس کیسے کیوں پھر کیونکہ شاید تاکہ تاکہ والا والے والی والوں تک تم ہم تمہارے ہمارا ہماری ہمارے میری میرا میرے آپ آپکا آپکی آپکے وہ ان انہیں انہوں انکے انکی انکا اسے اسے اسے اسکا اسکی اسکے اسی اسی اس میں تھا تھے تھیں تھوڑی تھوڑا تھوڑے زیادہ سب کوئی کئی کچھ یہ وہ وہی یہیں یہاں وہاں کیسے کیسا کسی کو نہ ہر وغیرہ علاوہ بعد پہلے علاوہ مز مزے مزہ لگا لگے لگا لگا ہو ہوئ ہوئ ہوئیں ہوئے ہوا ہوجائے ہوگئے ہوگئی ہوجاتی ہوچکا ہوچکی ہوچکے ہو رہا ہو رہی ہو رہے
""".split())

# Treebank Tokenizer for English
--- Tokenizer specifically designed fro English syntax (punctuation, contractions)

In [7]:
en_tokenizer = TreebankWordTokenizer()

# Cleaners

In [8]:
def clean_english(text):
    if pd.isna(text): return text #check if text is missing 
    words = en_tokenizer.tokenize(text.lower()) #convert to lowercase, use treebank tokenizer to split text into tokens
    words = [word for word in words if word.isalpha()] #keep only alphabetic words
    return ' '.join([w for w in words if w not in nltk_stopwords_dict['en']]) #remove stopwords and return clean text as string 

def clean_mandarin(text):
    if pd.isna(text): return text
    words = jieba.lcut(str(text)) # Uses jieba to segment Chinese text into words. Chinese has no spaces
    return ' '.join([word for word in words if word not in extended_mandarin_stopwords and word.strip()])

#Handles all other languages 
def generic_cleaner(text, lang):
    if pd.isna(text): return text
    words = re.findall(r'\b\w+\b', text.lower())
    
    if lang == 'ar': #Arabic, Mandarin, Farsi, and Urdu get extended manual lists
        stopword_set = extended_arabic_stopwords
    elif lang == 'zh':
        stopword_set = extended_mandarin_stopwords
    elif lang == 'fa':
        stopword_set = extended_farsi_stopwords
    elif lang == 'ur':
        stopword_set = extended_urdu_stopwords
    elif lang in nltk_stopwords_dict: #NLTK stopwords for known languages: Spanish, Porugese, French, Russian, turkish, indonesian
        stopword_set = nltk_stopwords_dict[lang]

    #Any fallback language uses the stopwords package ( Ukranian, Hindi)
    else:
        try:
            stopword_set = set(get_stopwords(lang))
        except:
            stopword_set = set()

    return ' '.join([w for w in words if w not in stopword_set])

def multilingual_cleaner(row):
    lang = row['language_code']
    text = row['messages']
    if lang == 'en':
        return clean_english(text)
    elif lang == 'zh':
        return clean_mandarin(text)
    else:
        return generic_cleaner(text, lang)

# --- Apply to DataFrame ---

In [9]:
df['messages_cleaned'] = df.apply(multilingual_cleaner, axis=1)

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/tp/9tzgnnj91lx4sd07z278clv40000gn/T/jieba.cache
Loading model cost 0.879 seconds.
Prefix dict has been built successfully.


In [10]:
languages_to_preview = ['en', 'ar', 'zh', 'fa', 'ru', 'es', 'tr', 'uk', 'ur', 'fr', 'hi', 'pt', 'id']

for lang in languages_to_preview:
    print(f"🔤 Language: {lang.upper()}")
    subset = df[df['language_code'] == lang][['messages', 'messages_cleaned']].head(3)
    print(subset.to_string(index=False))  # prettier printing without row numbers
    print("-" * 60)


🔤 Language: EN
                                                                                                             messages                                      messages_cleaned
I have been using alcohol to numb my pain, but I know it’s not the answer. How do I find healing through God instead? using alcohol numb pain know find healing god instead
                              I have been through detox before, but I always go back. How do I make a lasting change?                   detox always go make lasting change
                                     I feel like my past defines me. How do I embrace the future that God has for me?             feel like past defines embrace future god
------------------------------------------------------------
🔤 Language: AR
                                                                                                       messages                                                                             messages_cleaned
لقد كنت أعاني من

# Export 


In [ ]:

df.to_csv("GMO_Cleaned_Messages_Final.csv", index=False)

#-- Checking for Errors

In [ ]:
print(df['language_code'].value_counts())


In [ ]:
print(df_ru.columns)